# Story 3 — How Punk Killed the Reverential Cover

Measures how the *approach* to covering Dylan has changed over time, using duration deviation from Dylan's originals as a "fidelity" proxy.

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

DATA_DIR = Path('..') / 'data'

covers = pd.read_csv(DATA_DIR / 'dylan_covers_with_popularity.csv')
recs = pd.read_csv(DATA_DIR / 'dylan_recordings.csv')
enriched = pd.read_csv(DATA_DIR / 'dylan_covers_enriched.csv')

print(f'Covers: {len(covers)}, Recordings: {len(recs)}')

In [ ]:
# Dylan's original durations as baselines (median per work)
dylan_orig = recs[recs['is_bob_dylan'] == True].groupby('work_title')['recording_length_ms'].median().dropna()
print(f'Works with Dylan duration baseline: {len(dylan_orig)}')

# Prep covers
covers['year'] = pd.to_datetime(covers['spotify_release_date'], errors='coerce').dt.year
covers['duration_ms'] = covers['spotify_duration_ms'].fillna(covers['recording_length_ms'])
covers = covers.merge(dylan_orig.rename('dylan_duration_ms').reset_index(), on='work_title', how='left')
covers['duration_ratio'] = covers['duration_ms'] / covers['dylan_duration_ms']
covers['abs_deviation'] = (covers['duration_ratio'] - 1).abs()

# Filter to valid data
valid = covers[
    (covers['year'].notna()) & 
    (covers['duration_ratio'].notna()) & 
    (covers['duration_ratio'] > 0.1) & 
    (covers['duration_ratio'] < 10)
].copy()
print(f'Valid covers with year + duration: {len(valid)}')

In [ ]:
# Artist genre classification
ARTIST_GENRE = {}

for a in ['Joan Baez', 'Judy Collins', 'Peter, Paul and Mary', 'Pete Seeger',
          'Odetta', 'Richie Havens', 'Donovan', 'Gordon Lightfoot',
          'Nana Mouskouri', 'Hugues Aufray', 'The Kingston Trio', 'The Seekers',
          'Fairport Convention', 'Sandy Denny', 'Richard Thompson',
          'Fleet Foxes', 'Iron & Wine', 'Sufjan Stevens', 'Bon Iver',
          'Barb Jungr', 'Barbara Dickson', 'Melanie', 'Jimmy LaFave',
          'Tracy Chapman', 'Ani DiFranco', 'Billy Bragg', 'Lucinda Williams',
          'The Avett Brothers', 'Mumford & Sons', 'Laura Marling', 'First Aid Kit']:
    ARTIST_GENRE[a] = 'Folk'

for a in ['The Byrds', 'Jimi Hendrix', 'The Rolling Stones', 'The Band',
          'Eric Clapton', 'Manfred Mann', "Manfred Mann's Earth Band",
          'George Harrison', 'Traveling Wilburys', 'Neil Young',
          'Tom Petty and the Heartbreakers', 'Tom Petty', 'Rod Stewart',
          'Bryan Ferry', 'Joe Cocker', 'Roger McGuinn', 'The Hollies',
          'Bruce Springsteen', 'Van Morrison', 'Johnny Winter',
          'Leon Russell', 'Cher', 'Stevie Nicks']:
    ARTIST_GENRE[a] = 'Classic Rock'

for a in ['Grateful Dead', 'Jerry Garcia Band, Jerry Garcia', 'Dead & Company',
          'Phish', 'Dave Matthews Band', 'Widespread Panic']:
    ARTIST_GENRE[a] = 'Jam Band'

for a in ['Patti Smith', 'The Clash', 'Joe Strummer', 'The Ramones',
          'Television', 'Talking Heads', 'Blondie', 'Elvis Costello',
          'The Pretenders', 'Siouxsie and the Banshees', 'Siouxsie',
          'Joy Division', 'New Order', 'The Cure', 'Wire',
          'Hüsker Dü', 'Sonic Youth', 'Black Flag', 'Dead Kennedys',
          'Misfits', 'Bad Religion', 'NOFX', 'Rancid',
          'Social Distortion', 'Green Day', 'Rise Against',
          'My Chemical Romance', 'Rage Against the Machine',
          'The Gaslight Anthem', 'The Hold Steady', 'Against Me!']:
    ARTIST_GENRE[a] = 'Punk / Post-Punk'

for a in ['Nirvana', 'Pearl Jam', 'Soundgarden', 'Alice in Chains',
          'Pixies', 'Dinosaur Jr', 'Pavement', 'Built to Spill',
          'Yo La Tengo', 'Modest Mouse', 'The National', 'Arcade Fire',
          'Radiohead', 'R.E.M.', 'Smashing Pumpkins',
          'PJ Harvey', 'Nick Cave', 'Jeff Buckley', 'Bright Eyes',
          'Cat Power', 'The White Stripes', 'Jack White',
          'Robyn Hitchcock', 'Mark Lanegan', 'The Decemberists',
          'Wilco', 'Beck', 'U2', 'Joan Osborne']:
    ARTIST_GENRE[a] = 'Alt / Indie'

for a in ['Johnny Cash', 'Waylon Jennings', 'Willie Nelson',
          'Emmylou Harris', 'Old Crow Medicine Show', 'Dolly Parton',
          'Glen Campbell', 'Bobby Bare', 'Charley Pride']:
    ARTIST_GENRE[a] = 'Country'

for a in ["Guns N' Roses", 'Judas Priest', 'Motörhead', 'Metallica',
          'Iron Maiden']:
    ARTIST_GENRE[a] = 'Metal / Hard Rock'

for a in ['Nina Simone', 'Cassandra Wilson', 'Madeleine Peyroux',
          'Jamie Cullum', 'Herbie Hancock']:
    ARTIST_GENRE[a] = 'Jazz'

for a in ['Elvis Presley', 'James Last', 'Nana Mouskouri']:
    ARTIST_GENRE[a] = 'Pop'

for a in ['Buddy Guy', 'Taj Mahal', 'Bonnie Raitt', 'Rory Gallagher']:
    ARTIST_GENRE[a] = 'Blues'

def classify_artist(name):
    if pd.isna(name): return None
    for known, genre in ARTIST_GENRE.items():
        if known.lower() in name.lower(): return genre
    return None

valid['artist_genre'] = valid['spotify_artist_name'].apply(classify_artist)
classified = valid[valid['artist_genre'].notna()]
print(f'Classified: {len(classified)} ({len(classified)/len(valid):.1%})')
print(classified['artist_genre'].value_counts())

In [ ]:
# === CHART 1: Fidelity over time (rolling median + spread) ===
decade_stats = valid.groupby(valid['year'].astype(int)).agg(
    n=('recording_id', 'count'),
    median_ratio=('duration_ratio', 'median'),
    q25=('duration_ratio', lambda x: x.quantile(0.25)),
    q75=('duration_ratio', lambda x: x.quantile(0.75)),
    pct_faithful=('abs_deviation', lambda x: (x < 0.15).mean()),
    pct_radical=('abs_deviation', lambda x: (x > 0.5).mean()),
).reset_index()
decade_stats = decade_stats[decade_stats['n'] >= 10]
decade_stats.head(20)

In [ ]:
# === CHART 2: Genre interpretation spectrum ===
genre_spectrum = classified.groupby('artist_genre').agg(
    n=('recording_id', 'count'),
    median_ratio=('duration_ratio', 'median'),
    mean_ratio=('duration_ratio', 'mean'),
    q25=('duration_ratio', lambda x: x.quantile(0.25)),
    q75=('duration_ratio', lambda x: x.quantile(0.75)),
    median_deviation=('abs_deviation', 'median'),
    median_pop=('spotify_popularity', 'median'),
    pct_faithful=('abs_deviation', lambda x: (x < 0.15).mean()),
).sort_values('median_ratio').reset_index()
genre_spectrum

In [ ]:
# === CHART 3: Faithful vs Radical by decade ===
valid['decade'] = (valid['year'] // 10 * 10).astype(int)
decade_fidelity = valid.groupby('decade').agg(
    n=('recording_id', 'count'),
    pct_faithful=('abs_deviation', lambda x: (x < 0.15).mean()),
    pct_moderate=('abs_deviation', lambda x: ((x >= 0.15) & (x <= 0.5)).mean()),
    pct_radical=('abs_deviation', lambda x: (x > 0.5).mean()),
    iqr=('duration_ratio', lambda x: x.quantile(0.75) - x.quantile(0.25)),
).reset_index()
decade_fidelity

In [ ]:
# === CHART 4: Genre × Era heatmap ===
classified_wy = classified[classified['year'].notna()].copy()
classified_wy['era'] = pd.cut(classified_wy['year'],
    bins=[1959, 1969, 1979, 1989, 1999, 2009, 2027],
    labels=['1960s', '1970s', '1980s', '1990s', '2000s', '2010s+'])

heatmap_data = classified_wy.groupby(['era', 'artist_genre'], observed=True).agg(
    median_ratio=('duration_ratio', 'median'),
    n=('recording_id', 'count'),
).reset_index()

# Pivot for heatmap
pivot = heatmap_data.pivot(index='artist_genre', columns='era', values='median_ratio')
pivot

In [ ]:
# === Notable radical reinterpretations for gallery ===
# Deduplicate by spotify_track_id, pick most interesting ones
gallery_candidates = valid[
    (valid['spotify_track_id'].notna()) &
    (valid['spotify_popularity'] > 0)
].drop_duplicates(subset='spotify_track_id').copy()

# Get a mix: most radical + most popular radical
radical = gallery_candidates[gallery_candidates['abs_deviation'] > 0.4].copy()
radical['interest_score'] = radical['abs_deviation'] * 0.4 + (radical['spotify_popularity'] / 100) * 0.6
top_radical = radical.nlargest(25, 'interest_score')

print(f'Radical reinterpretations (>40% deviation, with Spotify): {len(radical)}')
for _, r in top_radical.head(20).iterrows():
    dur = r['duration_ms'] / 60000
    dylan = r['dylan_duration_ms'] / 60000
    print(f"  {r['spotify_artist_name']:35s} \"{r['work_title'][:35]:35s}\" ratio={r['duration_ratio']:.2f} ({dur:.1f}m vs {dylan:.1f}m) pop={r['spotify_popularity']:.0f}")

In [ ]:
# === Export JSON for HTML page ===

export = {}

# Chart 1: Fidelity timeline (year-level)
export['fidelity_timeline'] = decade_stats[['year', 'n', 'median_ratio', 'q25', 'q75', 
                                             'pct_faithful', 'pct_radical']].to_dict('records')

# Chart 2: Genre spectrum
export['genre_spectrum'] = genre_spectrum.to_dict('records')

# Chart 3: Decade fidelity breakdown
export['decade_fidelity'] = decade_fidelity.to_dict('records')

# Chart 4: Genre × Era heatmap
export['genre_era_heatmap'] = heatmap_data.to_dict('records')

# Gallery: top radical reinterpretations
gallery = []
for _, r in top_radical.head(20).iterrows():
    gallery.append({
        'artist': r['spotify_artist_name'],
        'song': r['work_title'],
        'spotify_name': r['spotify_track_name'],
        'duration_min': round(r['duration_ms'] / 60000, 1),
        'dylan_duration_min': round(r['dylan_duration_ms'] / 60000, 1),
        'ratio': round(r['duration_ratio'], 2),
        'popularity': int(r['spotify_popularity']),
        'spotify_url': r['spotify_external_url'] if pd.notna(r['spotify_external_url']) else None,
        'album': r['spotify_album_name'] if pd.notna(r['spotify_album_name']) else None,
        'year': int(r['year']) if pd.notna(r['year']) else None,
        'genre': classify_artist(r['spotify_artist_name']),
    })
export['gallery'] = gallery

# Summary stats
export['stats'] = {
    'total_covers_analyzed': len(valid),
    'total_with_genre': len(classified),
    'median_deviation_overall': round(valid['abs_deviation'].median(), 3),
    'pct_faithful_1980s': round(decade_fidelity[decade_fidelity['decade']==1980]['pct_faithful'].iloc[0], 3),
    'pct_faithful_2020s': round(decade_fidelity[decade_fidelity['decade']==2020]['pct_faithful'].iloc[0], 3),
    'iqr_1980s': round(decade_fidelity[decade_fidelity['decade']==1980]['iqr'].iloc[0], 3),
    'iqr_2020s': round(decade_fidelity[decade_fidelity['decade']==2020]['iqr'].iloc[0], 3),
    'jam_median_ratio': round(genre_spectrum[genre_spectrum['artist_genre']=='Jam Band']['median_ratio'].iloc[0], 2),
    'country_median_ratio': round(genre_spectrum[genre_spectrum['artist_genre']=='Country']['median_ratio'].iloc[0], 2),
}

with open('story3_data.json', 'w') as f:
    json.dump(export, f, indent=2, default=str)

print(f'Exported {len(export)} sections to story3_data.json')
print(f'Stats: {json.dumps(export["stats"], indent=2)}')